# **📘 Cleaning Employee Data for KLB Industries**

### **🎯 Objective**

You’re starting as a Data Engineer at KLB Industries (a defense contractor). Your first task is to clean and prepare an employee dataset for future use in a relational database.

---

### **📦 Dataset Overview**

🗂 File Name: `new_employee_data_messy.csv`

The dataset is a CSV file called **`new_employee_data_messy.csv`**. It was quickly thrown together by HR, so it might be quite a mess.

#### **It includes important fields such as:**

- 👤 **Name** – The name of an individual.  
- 🎓 **HighestDegree** – The highest academic degree the individual has obtained.  
- 🏢 **Division** – The division of KLB that the individual works in.  
- 🏢 **DivisionAddress** – The address of the KLB division where the individual works.  
- 🏠 **PersonalAddress** – The individual's personal address.  
- 📧 **Email** – The individual's email address. Clean every email to **`firstname_lastname_number@KLB.com`** (lowercase names, domain written exactly `KLB.com`), which is what the validator enforces. The raw domains are inconsistent (`klb.com`, `KLB.COM`, `klb..com`, `KLB..COM`).  
- 📞 **PhoneNumber** – The individual's phone number.  
- 💼 **Department** – The department in which the individual works.  
- 💼 **JobTitle** – The individual's job title.  
- 💼 **JobDescription** – The individual's job description.  
- 💰 **Salary** – The individual's annual salary.  
- 🎂 **Birthdate** – The individual's birthdate.  
- 🆔 **SocialSecurity** – The individual's Social Security Number (SSN).  

---

### **Your Current Task**

You need to **develop and apply data-cleaning techniques** to fix various issues in the dataset.

#### **🚧 Cleaning Tasks**
- 🔠 **Correct capitalization** and **remove extraneous special characters**.  
- ❓ **Fill missing values** where possible.  
- 🧼 **Validate and reformat fields** like SSN, email, and phone numbers.  
- 🔁 **Remove duplicate rows** (only if they are exact duplicates).  
- 🔗 **Ensure logical consistency** (e.g., job titles must align with departments).  

### **Final Step**

💾 **Export final version as** `Cleaned_data.csv` (exactly this name — the validator and Assignment 5 both look for `Cleaned_data.csv`).  

---

### 🚀 **Now, you're ready to transform messy employee data into a structured dataset!**


###  **🧰 Step 1: Read CSV file into pandas dataframe and inspect**

In [497]:
## Display in Notebook
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import pandas as pd
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_colwidth', None)  # Show full column width

import numpy as np
import os
from collections import Counter
import re

### **📥 Step 2: Load the CSV File**

This section loads the dataset from a CSV file into a Pandas DataFrame.

1. **File Path**:
   - Update the `csv_file` variable with the correct path to the file on your local machine.
   - If you're unsure about the file's location, you can right-click the file and copy its full path.

2. **Understanding the Data**:
   - `df.info()`: Provides a summary of the data structure, including column names, data types, and the number of missing values.
   - `df.Name.head(10)`: Displays the first 10 names from the dataset to ensure the data loaded properly.

In [498]:
# Define the path to your CSV file
# The CSV lives in the SAME folder as this notebook, so a relative path works on any machine.

csv_file = "new_employee_data_messy.csv"

try:
    # Attempt to read the CSV file
    df = pd.read_csv(csv_file)
    print("✅ Data loaded successfully!")
    df.info()  # Display DataFrame information

except FileNotFoundError:
    print(f"❌ Error: The file '{csv_file}' was not found.")

except pd.errors.EmptyDataError:
    print(f"❌ Error: The file '{csv_file}' is empty.")

except pd.errors.ParserError:
    print(f"❌ Error: The file '{csv_file}' contains malformed data.")

except Exception as e:
    print(f"⚠️ An unexpected error occurred: {e}")
    

✅ Data loaded successfully!
<class 'pandas.DataFrame'>
RangeIndex: 15752 entries, 0 to 15751
Data columns (total 16 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   EmployeeID             15667 non-null  float64
 1   Name                   15752 non-null  str    
 2   HighestDegree          15670 non-null  str    
 3   Division               15679 non-null  str    
 4   DivisionAddress        15698 non-null  str    
 5   DivisionPhone          15676 non-null  str    
 6   PersonalAddress        15678 non-null  str    
 7   Email                  15687 non-null  str    
 8   PhoneNumber            15676 non-null  str    
 9   Department             15690 non-null  str    
 10  DepartmentDescription  15692 non-null  str    
 11  JobTitle               15687 non-null  str    
 12  JobDescription         15662 non-null  str    
 13  Salary                 15687 non-null  float64
 14  Birthdate              15696 non-null

### **🔍 Step 3: Initial Data Inspection**

#### 👯‍♂️ Check for Duplicates and Drop Them


1. **Why Check for Duplicates?**
   - Duplicate rows can inflate metrics or bias results in analysis.
   - Removing duplicates ensures data integrity.

2. **Key Steps**:
   - `df.duplicated().sum()`: Counts how many rows are duplicates.
   - `df.drop_duplicates()`: Removes duplicate rows, keeping only the first occurrence.

This ensures the dataset is clean and contains only unique records.

In [499]:
# Count the number of duplicate rows
duplicate_count = df.duplicated().sum()

# Print the count of duplicates
print(f'📊 Total rows: {len(df)}')
print(f"🔁 Duplicate rows: {duplicate_count}")
df = df.drop_duplicates()
print(f'Number of unique rows: {len(df)}')

📊 Total rows: 15752
🔁 Duplicate rows: 1431
Number of unique rows: 14321


### **🧹 Step 4: Explore & Handle Missing Values**

We’ll look at:
- 🔍 Which columns have missing values?
- 🤔 How many values are missing?
- 🧠 Should we fill them or drop them?

1. **Why Standardize Missing Values?**
   - Different formats of missing values (`NaN`, `None`, empty strings) can cause confusion or errors during data processing.
   - Using `pd.NA` provides consistency across the dataset.

2. **Key Steps**:
   - `df.replace({np.nan: pd.NA})`: Converts all `NaN` values to `pd.NA`.
   - `df.isna().sum()`: Counts missing values for each column.
   - `df.isnull().any(axis=1).sum()`: Calculates how many rows have at least one missing value.

Use this information to decide how to handle missing values in the next steps (e.g., imputing, removing, or leaving as is). Run this cell and analyze your dataset!

In [500]:
# Identify and Standardize Missing Values

# Replace all NaN values with Pandas' NA representation
df = df.replace({np.nan: pd.NA})

# Count the number of missing values in each column
missing_values = df.isna().sum()

# Count the number of rows with at least one missing value
num_rows_with_missing = df.isnull().any(axis=1).sum()

# Display results
print(f" Number of Rows with Missing Values: {num_rows_with_missing}\n")

print(" Missing Values Count Per Column (Before Cleaning):")
print(missing_values)


 Number of Rows with Missing Values: 899

 Missing Values Count Per Column (Before Cleaning):
EmployeeID               80
Name                      0
HighestDegree            74
Division                 68
DivisionAddress          50
DivisionPhone            64
PersonalAddress          72
Email                    56
PhoneNumber              70
Department               55
DepartmentDescription    54
JobTitle                 62
JobDescription           81
Salary                   59
Birthdate                51
SocialSecurity           46
dtype: int64


### **🧼 Step 5: Data Cleaning and Feature Transformation**

To analyze the dataset effectively, we must ensure the data is clean and consistent. This step involves:
1. **Examining Each Feature**: Identify issues like inconsistent capitalization, special characters, or missing values.
2. **Applying Transformations**: Modify the data as needed to improve its quality.
3. **Splitting and Extracting**: Break complex fields (like `Name`) into meaningful components (e.g., First Name, Last Name).<br>
<br>


### **👤 Name Field Cleaning & Transformation**

- **Clean and Transform the `Name` Field**:
  - Fix capitalization and remove special characters.
  - Extract prefixes, suffixes, and degrees.
  - Split the name into components (e.g., Honorific, First Name, Last Name).<br>
<br>



In [501]:
#🧪 Inspect current state of the name column
df['Name'].head(50)

0                Steven Sean Johnson
1             Robert Matthew Schultz
2                Donna Brittany Ford
3     TERESA JENNIFER HERNANDEZ, THD
4            Christopher Perry Lopez
5             Maj Paul Carlos Guerra
6                 Martin David Brown
7           Catherine Rebecca Hurley
8           TONY KRISTOPHER MORRISON
9                 MICHAEL BARRY COOK
10        Genrl Kylie Donna Mitchell
11                   Martin John Cox
12               Kayla Amber Salazar
13                Robert Jeremy Wong
14           Margaret Karen Lee, ThD
15                 Justin Dean Brown
16                Lauren Mia Flowers
17              Autumn Joanna Coffey
18            Zachary William Miller
19            Tiffany Adriana Burton
20               James Richard Davis
21            KEITH DARREN RODRIGUEZ
22          Anthony Gregory Bell, BA
23           Edward Dalton Wilkerson
24              Joseph Timothy Smith
25              Kimberly Robin Boone
26              Tina Valerie Spencer
2

#### **🎩 Identify the honorifics in the data set**
We have to separate the honorifics from the first name. 
1. Split the name and look to see if the first element of the split is in a corpus of first names.
2. If not, it is an honorific.

#### **📥 Load First Name Corpus**
1. Got to https://www.ssa.gov/oact/babynames/limits.html
2. Click 'National Data'
3. It will download a compressed file of baby names from 1880's to now.
4. Extract this into a `data/first_names` folder **next to this notebook** (the code below creates `data/first_names` for you — just extract the txt files into it).
5. Note there will be about 145 txt files (`yob1880.txt` onward); the code below reads `yob1880`–`yob2022`

> ⚠️ **Note:** `names.zip` is bot-blocked for scripts — automated downloads (requests/curl) will fail. Download it via your browser from ssa.gov, or use the copy staged on Canvas.


In [502]:

# Folder where SSA files are extracted (relative to this notebook)
ssa_folder = "data/first_names"
os.makedirs(ssa_folder, exist_ok=True)  # creates the folder if it doesn't exist — extract the SSA txt files here

# Initialize an empty set for all names
ssa_names = set()

# Loop through all yobYYYY.txt files
for year in range(1880, 2023):  # From 1880 to 2022
    file_path = os.path.join(ssa_folder, f"yob{year}.txt")
    df2 = pd.read_csv(file_path, names=['name', 'gender', 'count'])
    
    # Add names to the set
    ssa_names.update(df2['name'].str.lower())

# Test if a name exists in the full SSA dataset
test_name = "Yesenia"
if test_name.lower() in ssa_names:
    print(f"{test_name} is a valid first name.")
else:
    print(f"{test_name} is NOT in the dataset.")

# Test if a name exists in the full SSA dataset
test_name = "Jackie"
if test_name.lower() in ssa_names:
    print(f"{test_name} is a valid first name.")
else:
    print(f"{test_name} is NOT in the dataset.")


test_name = "XXX"
if test_name.lower() in ssa_names:
    print(f"✅ {test_name} is a valid first name.")
else:
    print(f"❌ {test_name} is NOT in the dataset.")

Yesenia is a valid first name.
Jackie is a valid first name.
❌ XXX is NOT in the dataset.


In [503]:

# 🧼 Function to clean and process the name
def process_name_first_token(name):
    # Remove special characters, periods, commas
    cleaned_name = re.sub(r'[^a-zA-Z\s]', '', name)  # Keep only letters and spaces
    cleaned_name = cleaned_name.strip()  # Remove leading/trailing whitespace
    
    # Split the name into parts
    name_parts = cleaned_name.split()
    
    # Return the first token or an empty string if the name is empty
    return name_parts[0] if name_parts else ''

# Initialize a set to store unique unmatched tokens
unmatched_tokens = set()

# Process each name in the DataFrame
for full_name in df['Name']:  # Note: Ensure column name matches your DataFrame
    first_token = process_name_first_token(full_name)  # Extract the first token
    # Normalize capitalization
    normalized_token = first_token.lower()
    # Check against the corpus and add to unmatched_tokens if not found
    if normalized_token and normalized_token not in ssa_names:
        unmatched_tokens.add(normalized_token)

# Output the unique unmatched tokens
print("Unique unmatched tokens (possible titles/honorifics):")
for token in sorted(unmatched_tokens):
    print(token)


Unique unmatched tokens (possible titles/honorifics):
adm
capt
cmdr
col
cpl
dr
genrl
hon
lt
maj
prof
prv
sgt


#### **🎓 Create Honorific and Degree Lists**
#### At this point we have two lists. 
1. **honorifics** ~ titles and appearing before the first name
2. **degrees_list** ~ degree acronymes occurring at the end <br>

Now, we want to split each name into its various parts, and create new dataframe columns **'Honorific'**, **'First Name'**,**'Middle Name'**, **'Last Name'**, **'Degree'**

In [504]:
# Convert unmatched_tokens to a sorted list
honorifics_list = sorted(unmatched_tokens)  

# Capitalize each honorific correctly
honorifics_list = [honorific.capitalize() for honorific in honorifics_list]

degree_list = ['PhD', 'MD', 'JD', 'DO', 'DDS', 'DVM', 'EdD', 'LLD', 'ScD', 'ThD',
                   'MBA', 'MS', 'MA', 'MSc', 'MEd', 'MFA', 'MLA', 'LLM',
                   'BS', 'BA', 'BSc', 'BFA', 'LLB', 'AB'
                  ]

# Print the formatted list
print(honorifics_list)
print(degree_list)

['Adm', 'Capt', 'Cmdr', 'Col', 'Cpl', 'Dr', 'Genrl', 'Hon', 'Lt', 'Maj', 'Prof', 'Prv', 'Sgt']
['PhD', 'MD', 'JD', 'DO', 'DDS', 'DVM', 'EdD', 'LLD', 'ScD', 'ThD', 'MBA', 'MS', 'MA', 'MSc', 'MEd', 'MFA', 'MLA', 'LLM', 'BS', 'BA', 'BSc', 'BFA', 'LLB', 'AB']


#### **🔎 Cleaning and Splitting the `Name` Field**

The `Name` column contains full names, including prefixes (e.g., "Dr."), suffixes (e.g., "Jr."), and degree titles (e.g., "PhD"). To analyze the data effectively, we need to:
1. **Fix Issues**:
   - Remove unnecessary special characters.
   - Ensure consistent capitalization.
   - Handle missing or empty names gracefully.
2. **Extract Components**:
   - Split the name into meaningful parts:
     - **Honorific**: e.g., "Dr.", "Prof."
     - **First Name**: The first part of the name.
     - **Middle Name(s)**: All names between the first and last name.
     - **Last Name**: The final part of the name.
     - **Degree**: e.g., "PhD", "MBA".
3. **Output Example**:
   - Input: `"Dr. John A. Smith, PhD"`
   - Output:
     ```
     Honorific: Dr
     First Name: John
     Middle Name: A
     Last Name: Smith
     Degree: PHD
     ```
4. **Regular Expressions (Regex)**:
   - We'll use regex to identify and extract patterns like honorifics and suffixes. If you're new to regex, check out:
     - [Regex Tutorial #1](https://www.youtube.com/watch?v=EzeeypMKx7o)
     - [Regex Tutorial #2](https://www.youtube.com/watch?v=j6A28L6Tmxw)


#### **🏷️ Handle the Prefixes and Suffixes**

To clean and process the `Name` column, we need to:
1. **Split Prefixes and Suffixes**:
   - Extract honorifics (e.g., "Dr.", "Prof.") into a separate column.
   - Extract degree suffixes (e.g., "PhD", "MBA").
2. **Clean the Remaining Name**:
   - Normalize capitalization.
   - Remove special characters.
   - Split the remaining name into `First Name`, `Middle Name`, and `Last Name`.
3. **Update Columns**:
   - Add columns for `Honorific`, `First Name`, `Middle Name`, `Last Name`, and `Degree`.

This ensures the `Name` column is fully processed into clean, structured components.

In [505]:
import pandas as pd
import re

# Step 1: Define a function to clean and process names
def process_name(full_name):
    # Step 1a: Remove commas and normalize spaces
    name = re.sub(r'[,\s]+', ' ', full_name.strip())  # Replace commas and extra spaces with a single space

    # Patterns for matching honorifics, name suffixes, and degree suffixes
    honorifics_pattern = r'\b(?:' + '|'.join(re.escape(h) for h in honorifics_list) + r')\b'
    degree_suffix_pattern = r'\b(?:' + '|'.join(re.escape(d) for d in degree_list) + r')\b'

    # FIX: Ensure degrees are fully removed BEFORE further processing
    prefix_remove = r'\b(?:Mr|Mrs|Ms|Miss)\.?\b'  # Remove non-honorific prefixes

    # Step 2: Extract and remove degree suffix (ensure full removal)
    degree_match = re.search(degree_suffix_pattern, name, flags=re.IGNORECASE)
    degree = degree_match.group(0).upper() if degree_match else ''
    
    if degree:
        name = re.sub(r'\b' + re.escape(degree) + r'\b', '', name, flags=re.IGNORECASE).strip()  # FORCE REMOVE DEGREE

    # Step 3: Remove non-honorific prefixes (e.g., "Mr.", "Mrs.")
    name = re.sub(prefix_remove, '', name, flags=re.IGNORECASE).strip()

    # Step 4: Extract and remove honorifics from the name
    honorific_match = re.search(honorifics_pattern, name, flags=re.IGNORECASE)
    honorific = honorific_match.group(0).capitalize() if honorific_match else ''
    name = re.sub(honorifics_pattern, '', name, flags=re.IGNORECASE).strip()

    # Step 5: Ensure spaces are normalized after removals
    name = re.sub(r'\s+', ' ', name).strip()  # Remove extra spaces again

    # Step 6: Split the cleaned name into parts
    parts = name.split()

    # Step 7: Assign first, middle, and last names
    first_name = parts[0].capitalize() if len(parts) > 0 else ''
    last_name = parts[-1].capitalize() if len(parts) > 1 else ''
    middle_name = ' '.join(p.capitalize() for p in parts[1:-1]) if len(parts) > 2 else ''

    return pd.Series([honorific, first_name, middle_name, last_name, degree])


# Step 2: Apply the function to the 'Name' column in the DataFrame
df[['Honorific', 'First Name', 'Middle Name', 'Last Name', 'Degree']] = df['Name'].apply(process_name)

In [506]:
# Step 3: Display the processed DataFrame in Jupyter Notebook
from IPython.display import display
display(df[['Name', 'Honorific', 'First Name', 'Middle Name', 'Last Name', 'Degree', "HighestDegree"]].head(10))

,Name,Honorific,First Name,Middle Name,Last Name,Degree,HighestDegree
0,Steven Sean Johnson,,Steven,Sean,Johnson,,MA
1,Robert Matthew Schultz,,Robert,Matthew,Schultz,,CTE
2,Donna Brittany Ford,,Donna,Brittany,Ford,,AB
3,"TERESA JENNIFER HERNANDEZ, THD",,Teresa,Jennifer,Hernandez,THD,MA
4,Christopher Perry Lopez,,Christopher,Perry,Lopez,,AB
5,Maj Paul Carlos Guerra,Maj,Paul,Carlos,Guerra,,HS
6,Martin David Brown,,Martin,David,Brown,,CTE
7,Catherine Rebecca Hurley,,Catherine,Rebecca,Hurley,,MA
8,TONY KRISTOPHER MORRISON,,Tony,Kristopher,Morrison,,HS
9,MICHAEL BARRY COOK,,Michael,Barry,Cook,,PhD


#### **Observations**
1. Special characters have been removed, and capitalization has been normalized across the dataset.
2. In some cases, the degree extracted from the `Name` column (`Degree`) does not match the value in the `HighestDegree` column.
   - For example, a name might contain "PhD" while `HighestDegree` lists "Master's."
3. To maintain consistency:
   - If the `Degree` column is not empty and its value differs from `HighestDegree`, we will replace the value in `HighestDegree` with the one from `Degree`.
4. Once the `HighestDegree` column is updated, the `Degree` column will be dropped to avoid redundancy.
5. Some degrees do not match the honorifics. For example, doctors will have some type of advanced degree not GED.
6. If a person has one of these honorifics \[Adm, Capt, Cmdr, Col, Dr, Genrl, Hon, Lt, Maj, Prof,\] and one of these highest degrees 
\[AB, HS, CTE, GED\], replace highest degree with 'Bachelors+'

#### **Let's look more closely**

In [507]:
# Set display options for better alignment
pd.set_option('display.colheader_justify', 'center')  # Center column headers
pd.set_option('display.max_columns', None)  # Ensure all columns are shown
pd.set_option('display.width', 1000)  # Expand width for better readability

print(df[['Name', 'Honorific', 'First Name', 'Middle Name', 'Last Name', 'Degree', 'HighestDegree']].head(10).to_string(index=False))


             Name              Honorific  First Name Middle Name Last Name Degree HighestDegree
           Steven Sean Johnson                Steven        Sean   Johnson              MA     
        Robert Matthew Schultz                Robert     Matthew   Schultz             CTE     
           Donna Brittany Ford                 Donna    Brittany      Ford              AB     
TERESA JENNIFER HERNANDEZ, THD                Teresa    Jennifer Hernandez  THD         MA     
       Christopher Perry Lopez           Christopher       Perry     Lopez              AB     
        Maj Paul Carlos Guerra    Maj           Paul      Carlos    Guerra              HS     
            Martin David Brown                Martin       David     Brown             CTE     
      Catherine Rebecca Hurley             Catherine     Rebecca    Hurley              MA     
      TONY KRISTOPHER MORRISON                  Tony  Kristopher  Morrison              HS     
            MICHAEL BARRY COOK          

In [508]:
import pandas as pd
import numpy as np

# Convert pd.NA back to np.nan for consistency
df = df.replace({pd.NA: np.nan})

# Define honorifics that should not have a low-level degree
advanced_honorifics = {'Adm', 'Capt', 'Cmdr', 'Col', 'Dr', 'Genrl', 'Hon', 'Lt', 'Maj', 'Prof'}

# Define degrees that should be replaced if paired with an advanced honorific
low_degrees = {'AB', 'HS', 'CTE', 'GED'}

# Define function to update HighestDegree based on Degree
def update_highest_degree(row):
    """
    Updates 'HighestDegree' with 'Degree' if 'Degree' is not empty 
    and 'HighestDegree' is either empty or different.
    """
    if pd.notna(row["Degree"]) and row["Degree"].strip():  # Ensure 'Degree' is not missing or empty
        if pd.isna(row["HighestDegree"]) or row["Degree"].strip() != row["HighestDegree"].strip():
            return row["Degree"].strip()  # Replace 'HighestDegree' with 'Degree'
    return row["HighestDegree"]  # Retain the original 'HighestDegree'

# Apply function to update 'HighestDegree'
df["HighestDegree"] = df.apply(update_highest_degree, axis=1)

# Step 2: Replace low degrees with 'Bachelors+' if honorific is in the advanced list
df.loc[df['Honorific'].isin(advanced_honorifics) & df['HighestDegree'].isin(low_degrees), 'HighestDegree'] = 'Bachelors+'

# # Step 3: Drop the redundant 'Degree' column (only if it exists)
# if 'Degree' in df.columns:
#     df.drop(columns=['Degree'], inplace=True)

# Print formatted table
print(df[['Name', 'Honorific', 'First Name', 'Middle Name', 'Last Name', 'Degree', 'HighestDegree']].head(100).to_string(index=False))



             Name              Honorific  First Name Middle Name Last Name Degree HighestDegree
           Steven Sean Johnson                Steven        Sean   Johnson                  MA 
        Robert Matthew Schultz                Robert     Matthew   Schultz                 CTE 
           Donna Brittany Ford                 Donna    Brittany      Ford                  AB 
TERESA JENNIFER HERNANDEZ, THD                Teresa    Jennifer Hernandez  THD            THD 
       Christopher Perry Lopez           Christopher       Perry     Lopez                  AB 
        Maj Paul Carlos Guerra     Maj          Paul      Carlos    Guerra          Bachelors+ 
            Martin David Brown                Martin       David     Brown                 CTE 
      Catherine Rebecca Hurley             Catherine     Rebecca    Hurley                  MA 
      TONY KRISTOPHER MORRISON                  Tony  Kristopher  Morrison                  HS 
            MICHAEL BARRY COOK          

In [509]:
## Look at honorifics and degrees
# Display rows where 'Degree' column is not empty (not NaN)
df_with_honorific = df[df['Honorific'].notna() & (df['Honorific'] != '')]

# Display the first 50 matching rows
print(df_with_honorific[['Name', 'Honorific', 'First Name', 'Middle Name', 'Last Name', 'Degree', 'HighestDegree']].head(50))

# Display the last 50 matching rows
print(df_with_honorific[['Name', 'Honorific', 'First Name', 'Middle Name', 'Last Name', 'Degree', 'HighestDegree']].tail(50))


                  Name              Honorific  First Name  Middle Name Last Name Degree HighestDegree
5            Maj Paul Carlos Guerra      Maj          Paul      Carlos    Guerra          Bachelors+ 
10       Genrl Kylie Donna Mitchell    Genrl         Kylie       Donna  Mitchell          Bachelors+ 
36       Hon Daniel Nicholas Parker      Hon        Daniel    Nicholas    Parker                  BS 
54          MAJ AMY KENDRA HOLLOWAY      Maj           Amy      Kendra  Holloway          Bachelors+ 
85          Dr Matthew Timothy Hess       Dr       Matthew     Timothy      Hess          Bachelors+ 
93           Prof Alan Richard Chen     Prof          Alan     Richard      Chen          Bachelors+ 
101              PRV AMY ANN GUERRA      Prv           Amy         Ann    Guerra                  HS 
119    Lt Deborah Stephanie Cabrera       Lt       Deborah   Stephanie   Cabrera                  BS 
155    Genrl Angela Veronica Briggs    Genrl        Angela    Veronica    Briggs  

In [510]:

df.info()

<class 'pandas.DataFrame'>
Index: 14321 entries, 0 to 15242
Data columns (total 21 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   EmployeeID             14241 non-null  object
 1   Name                   14321 non-null  str   
 2   HighestDegree          14250 non-null  str   
 3   Division               14253 non-null  str   
 4   DivisionAddress        14271 non-null  str   
 5   DivisionPhone          14257 non-null  str   
 6   PersonalAddress        14249 non-null  str   
 7   Email                  14265 non-null  str   
 8   PhoneNumber            14251 non-null  str   
 9   Department             14266 non-null  str   
 10  DepartmentDescription  14267 non-null  str   
 11  JobTitle               14259 non-null  str   
 12  JobDescription         14240 non-null  str   
 13  Salary                 14262 non-null  object
 14  Birthdate              14270 non-null  str   
 15  SocialSecurity         14275 non-nu

#### **🏢 Division**

The `Division` column contains the names of divisions where employees work. However, these names may:
1. Contain special characters.
2. Have inconsistent capitalization.
3. Include invalid or unrecognized division names.
4. Have missing values.

To address these issues, we will:
1. Retain missing values as they are.
2. Remove special characters from division names.
3. Normalize capitalization to title case.
4. Verify that each division name matches a predefined list of valid divisions.
5. Missing or unrecognized divisions will be set to `NaN`.



In [511]:
# Step 1: Explore the Division column
# - View the first 20 rows of the Division and DivisionAddress columns
print("First 20 rows of the 'Division' column:")
print(df.Division.head(20))

First 20 rows of the 'Division' column:
0       South Division
1       North Division
2       North Division
3       South Division
4        West Division
5        West Division
6        West Division
7        East Division
8        WEST DIVISION
9                  NaN
10       East Division
11      SOUTH DIVISION
12       West Division
13      SOUTH DIVISION
14      North Division
15       East Division
16      North Division
17      SOUTH DIVISION
18    CENTRAL DIVISION
19    Central Division
Name: Division, dtype: str


In [512]:
# Step 2: Define a list of valid divisions
# - This list contains all acceptable division names

# 'Corporate' is NOT valid: those rows are set to missing here and get their division back from
# DivisionAddress (456 South Ave = South Division) in the fill step further down.
Divisions = ['North Division', 'South Division', 'East Division', 'West Division', 'Central Division']

# Step 3: Define a function to clean and validate the Division column
def clean_division(division):
    # Retain NaN values
    if pd.isna(division):  # Check if the value is NaN
        return division  # Return NaN as it is

    # Step 3.1: Convert to string (if not already)
    division = str(division)

    # Step 3.2: Remove special characters
    division = re.sub(r'[^a-zA-Z0-9\s,]', '', division)

    # Step 3.3: Normalize capitalization (convert to title case)
    division = division.title()

    # Step 3.4: Validate against the predefined list
    if division not in Divisions:
        print(f"Unrecognized division: {division}")  # Debug output for unrecognized divisions
        return pd.NA  # Mark as missing if not valid
    
    return division  # Return the cleaned division name

    

> **The validator accepts only the 5 geographic divisions.** The few `Corporate` rows will print as *Unrecognized division* below and become missing; that is expected. Their `DivisionAddress` is the South Division address, so the **Fill Missing `Division`** step below restores them as `South Division`.


In [513]:
# Step 4: Apply the cleaning function to the 'Division' column
df["Division"] = df["Division"].apply(clean_division)

# Step 5: Check for missing values in the Division column
missing_division = df["Division"].isna().sum()
print(f"\nMissing values in 'Division': {missing_division}")

# Step 6: Display the first 10 rows of the CleanedDivision column
print("\nFirst 10 rows of 'Division':")
print(df.Division.head(10))

Unrecognized division: Corporate
Unrecognized division: Corporate
Unrecognized division: Corporate
Unrecognized division: Corporate
Unrecognized division: Corporate

Missing values in 'Division': 73

First 10 rows of 'Division':
0    South Division
1    North Division
2    North Division
3    South Division
4     West Division
5     West Division
6     West Division
7     East Division
8     West Division
9               NaN
Name: Division, dtype: str


#### **🏠 Divison Addresses**

The `DivisionAddress` column contains the addresses associated with each division. These addresses may:
1. Contain unnecessary special characters.
2. Have inconsistent capitalization.
3. Include improperly formatted state abbreviations.

To address these issues, we will:
1. Retain missing values as they are.
2. Remove special characters (except spaces).
3. Normalize capitalization (e.g., "123 MAIN st" → "123 Main St").
4. Ensure state abbreviations are correctly capitalized (e.g., "tx" → "TX").

We will use a predefined list of valid state abbreviations to ensure accuracy.


In [514]:
df.DivisionAddress.head(20)

0         456 SOUTH AVE, AUSTIN, TX 73301
1     123 NORTH ST, SPRINGFIELD, IL 62701
2     123 North St, Springfield, IL 62701
3         456 SOUTH AVE, AUSTIN, TX 73301
4       101 WEST DR, SACRAMENTO, CA 94203
5       101 West Dr, Sacramento, CA 94203
6       101 West Dr, Sacramento, CA 94203
7         789 East Blvd, Albany, NY 12201
8       101 WEST DR, SACRAMENTO, CA 94203
9                                     NaN
10        789 East Blvd, Albany, NY 12201
11        456 South Ave, Austin, TX 73301
12      101 West Dr, Sacramento, CA 94203
13        456 South Ave, Austin, TX 73301
14    123 NORTH ST, SPRINGFIELD, IL 62701
15        789 EAST BLVD, ALBANY, NY 12201
16    123 North St, Springfield, IL 62701
17        456 SOUTH AVE, AUSTIN, TX 73301
18        202 Main St, Columbus, OH 43085
19        202 MAIN ST, COLUMBUS, OH 43085
Name: DivisionAddress, dtype: str

##### **📍 Reference — Correct addresses and state abbreviations given below**

In [515]:
# Step 1: Define the correct addresses for each division
Division_Addresses = [
    {
        'division': 'North Division',
        'address': '123 North St, Springfield, IL 62701',
        'city': 'Springfield',
        'state': 'IL',
        'zip_code': '62701',
        'phone': '(217)555-1234'
    },
    {
        'division': 'South Division',
        'address': '456 South Ave, Austin, TX 73301',
        'city': 'Austin',
        'state': 'TX',
        'zip_code': '73301',
        'phone': '(512)555-6789'
    },
    {
        'division': 'East Division',
        'address': '789 East Blvd, Albany, NY 12201',
        'city': 'Albany',
        'state': 'NY',
        'zip_code': '12201',
        'phone': '(518)555-1234'
    },
    {
        'division': 'West Division',
        'address': '101 West Dr, Sacramento, CA 94203',
        'city': 'Sacramento',
        'state': 'CA',
        'zip_code': '94203',
        'phone': '(916)555-5678'
    },
    {
        'division': 'Central Division',
        'address': '202 Main St, Columbus, OH 43085',
        'city': 'Columbus',
        'state': 'OH',
        'zip_code': '43085',
        'phone': '(916)555-6789'
    }
]


# List of all valid state abbreviations
STATE_ABBREVIATIONS = [
    "AL", "AK", "AZ", "AR", "CA", "CO", "CT", "DE", "FL", "GA", "HI", "ID", "IL", "IN", "IA", "KS", 
    "KY", "LA", "ME", "MD", "MA", "MI", "MN", "MS", "MO", "MT", "NE", "NV", "NH", "NJ", "NM", "NY", 
    "NC", "ND", "OH", "OK", "OR", "PA", "RI", "SC", "SD", "TN", "TX", "UT", "VT", "VA", "WA", "WV", 
    "WI", "WY"
]

# Step 2: Create a regex pattern to match state abbreviations
# - Matches any valid two-letter state abbreviation (case-insensitive)
STATE_PATTERN = r'\b(' + '|'.join(STATE_ABBREVIATIONS) + r')\b'
# Display the regex pattern (for debugging purposes)
print(STATE_PATTERN)

\b(AL|AK|AZ|AR|CA|CO|CT|DE|FL|GA|HI|ID|IL|IN|IA|KS|KY|LA|ME|MD|MA|MI|MN|MS|MO|MT|NE|NV|NH|NJ|NM|NY|NC|ND|OH|OK|OR|PA|RI|SC|SD|TN|TX|UT|VT|VA|WA|WV|WI|WY)\b


In [516]:
# Step 3: Define a function to clean and normalize addresses
def clean_address(address):
    # Ensure the input is a string and handle NaN values
    if pd.isna(address):  # Check if the value is NaN
        return address  # Return NaN as it is
    address = str(address)  # Convert to string if not already

    # Remove special characters (except spaces)
    address = re.sub(r'[^a-zA-Z0-9\s,]', '', address)

    # Remove commas
    address = address.replace(',', '')

    # Normalize capitalization (title case for the address)
    address = address.title()  # note that this turns TX in Tx, so we must correct

    # Find and capitalize the state abbreviation
    def capitalize_state(match):
        return match.group(0).upper()

    # Apply the regex pattern to capitalize state abbreviations
    # Note that STATE_PATTERN is matched against address, the match is sent to function capitalize_state, 
    # where is gets converted to all caps, then the substituted into the string address.
    address = re.sub(STATE_PATTERN, capitalize_state, address, flags=re.IGNORECASE)

    return address

In [517]:
# Apply the function to clean the Address
df["DivisionAddress"] = df["DivisionAddress"].apply(clean_address)

# Check for missing values in CleanedDivisionAddress
missing_division_address = df["DivisionAddress"].isna().sum()
print(f"Missing values in DivisionAddress: {missing_division_address}")

df.DivisionAddress.head(15)

Missing values in DivisionAddress: 50


0         456 South Ave Austin TX 73301
1     123 North St Springfield IL 62701
2     123 North St Springfield IL 62701
3         456 South Ave Austin TX 73301
4       101 West Dr Sacramento CA 94203
5       101 West Dr Sacramento CA 94203
6       101 West Dr Sacramento CA 94203
7         789 East Blvd Albany NY 12201
8       101 West Dr Sacramento CA 94203
9                                   NaN
10        789 East Blvd Albany NY 12201
11        456 South Ave Austin TX 73301
12      101 West Dr Sacramento CA 94203
13        456 South Ave Austin TX 73301
14    123 North St Springfield IL 62701
Name: DivisionAddress, dtype: str

In [518]:
# Check for missing values in Division
# - This counts the number of rows where Division is NaN (missing).

missing_division = df["Division"].isna().sum()
print(f"Missing values in Division: {missing_division}")

# Check for missing values in DivisionAddress
# - This counts the number of rows where DivisionAddress is NaN (missing).

missing_division_address = df["DivisionAddress"].isna().sum()
print(f"Missing values in DivisionAddress: {missing_division_address}")

# Identify rows where both Division and DivisionAddress are missing
# - This creates a filtered DataFrame with rows where both columns are NaN.
missing_rows = df[df["Division"].isna() & df["DivisionAddress"].isna()]

# Display the number of rows with missing values in both columns
print(f"Number rows with missing values in both 'Division' and 'DivisionAddress': {len(missing_rows)}")
print("Rows with missing values in both 'Division' and 'DivisionAddress':")

print(missing_rows.index.tolist()) 

Missing values in Division: 73
Missing values in DivisionAddress: 50
Number rows with missing values in both 'Division' and 'DivisionAddress': 3
Rows with missing values in both 'Division' and 'DivisionAddress':
[9, 126, 196]


In [519]:
# Try to fill in missing values where both are missing

# Define mapping of states to divisions
state_to_division = {
    'IL': 'North Division',
    'TX': 'South Division',
    'NY': 'East Division',
    'CA': 'West Division',   
    'OH': 'Central Division'   
}

# Define mapping of divisions to addresses
division_address_map = {entry['division']: entry['address'] for entry in Division_Addresses}

# Function to update missing Division and DivisionAddress
def fix_missing_division(row):
    if pd.isna(row['Division']) and pd.isna(row['DivisionAddress']):
        # Extract state from PersonalAddress
        state_match = next((state for state in state_to_division if state in str(row['PersonalAddress'])), None)
        if state_match:
            division = state_to_division[state_match]
            division_address = division_address_map.get(division, None)
            row['Division'] = division
            row['DivisionAddress'] = division_address
    return row

# Apply the function to only the rows where both Division and DivisionAddress are missing
df.loc[df['Division'].isna() & df['DivisionAddress'].isna(), :] = df.loc[
    df['Division'].isna() & df['DivisionAddress'].isna()].apply(fix_missing_division, axis=1)

indices = [9, 126, 196]  # Replace with actual index values

# Print the Division and Division Address for the specified indices
print(df.loc[indices, ['Division', 'DivisionAddress']])


         Division             DivisionAddress         
9    Central Division  202 Main St, Columbus, OH 43085
126     East Division  789 East Blvd, Albany, NY 12201
196  Central Division  202 Main St, Columbus, OH 43085


#### **🧩 Fill Missing `DivisionAddress` (When `Division` Is Present)**

To handle rows with a missing `DivisionAddress`, we will:
1. Use the `Division` column to identify the division.
2. Fill in the missing `DivisionAddress` using a predefined mapping of divisions to their correct addresses (`division_to_address`).
3. Leave the `DivisionAddress` as is for rows where both the division and address are missing.

This step ensures that every division has a corresponding address wherever possible.

In [520]:
# Step 1: Create a mapping of divisions to their correct addresses
# - This uses the predefined Division_Addresses list to create a dictionary.
division_to_address = {entry['division']: entry['address'] for entry in Division_Addresses}

# Step 2: Define a function to fill missing division addresses
def fill_missing_division_address(row):
    
    # Step 2.1: Check if 'DivisionAddress' is missing
    if pd.isna(row['DivisionAddress']):
        # Step 2.2: Use 'Division' to look up the correct address
        return division_to_address.get(row['Division'], row['DivisionAddress'])
    
    # Step 2.3: Return the original address if it's not missing
    return row['DivisionAddress']


In [521]:
# Step 3: Apply the function to the DataFrame
# - This updates the 'DivisionAddress' column with filled values where possible.
df['DivisionAddress'] = df.apply(fill_missing_division_address, axis=1)

# Step 4: Check for remaining missing values in the 'DivisionAddress' column
missing_division_address = df["DivisionAddress"].isna().sum()
print(f"Missing values in 'DivisionAddress' after filling: {missing_division_address}")

# Step 5: Identify rows where 'Division' is present but 'DivisionAddress' is missing
rows_with_missing_address = df[df['DivisionAddress'].isna() & df['Division'].notna()]

# Display up to 5 rows showing 'Division' and 'DivisionAddress'
if not rows_with_missing_address.empty:
    print("\nRows with Division but missing DivisionAddress (showing up to 5):")
    print(rows_with_missing_address[['Division', 'DivisionAddress']].head(5).to_string(index=False))
    rows_with_missing_address
else:
    print("\nNo rows found where Division is present but DivisionAddress is missing.")


Missing values in 'DivisionAddress' after filling: 0

No rows found where Division is present but DivisionAddress is missing.


#### **🔄 Fill Missing `Division` (When `DivisionAddress` Is Present)**

In cases where the `Division` is missing but the `DivisionAddress` is available, we will:
1. Use the `DivisionAddress` column to identify the corresponding division.
2. Fill in the `Division` column using a predefined mapping of addresses to divisions (`address_to_division`).
3. Leave the `Division` as `NaN` if no valid address is found.

This step ensures that divisions are correctly filled based on their associated addresses.

In [522]:
# Step 1: Normalize and create a mapping of addresses to their corresponding divisions
address_to_division = {
    entry['address'].replace(",", "").strip().lower(): entry['division']
    for entry in Division_Addresses
}

# Step 2: Define a function to fill missing divisions based on the address
def fill_missing_division(row):
    # Step 2.1: If 'Division' is already present, return it as is
    if not pd.isna(row['Division']):
        return row['Division']

    # Step 2.2: If 'DivisionAddress' is missing, return NaN
    if pd.isna(row['DivisionAddress']):
        return pd.NA

    # Step 2.3: Normalize 'DivisionAddress' for lookup (strip spaces, remove commas, convert to lowercase)
    address = row['DivisionAddress'].replace(",", "").strip().lower()

    # Step 2.4: Use 'DivisionAddress' to look up the corresponding division
    if address in address_to_division:
        return address_to_division[address]

    # Step 2.5: Log unmatched addresses for debugging
    unmatched_addresses.add(address)

    # Step 2.6: Return NaN if no matching division is found
    return pd.NA

# # Step 3: Initialize a set to track unmatched addresses
# unmatched_addresses = set()

# # Step 4: Apply the function to the DataFrame
# df['Division'] = df.apply(fill_missing_division, axis=1)

# # Step 5: Debug unmatched addresses
# print(f"Number of unmatched addresses: {len(unmatched_addresses)}")
# if unmatched_addresses:
#     print("Sample unmatched addresses:", list(unmatched_addresses)[:10])  # Print first 10 for review


In [523]:

unmatched_addresses = set()  # used by fill_missing_division to log addresses it cannot match

# Step 3: Apply the function to the DataFrame
# - This updates the 'CleanedDivision' column with filled values where possible.
df['Division'] = df.apply(fill_missing_division, axis=1)

# Step 4: Check for remaining missing values in the CleanedDivision column
missing_division = df["Division"].isna().sum()
missing_division_address = df["DivisionAddress"].isna().sum()
print(f"Missing values in 'Division' after filling: {missing_division}")
print(f"Missing values in 'DivisionAddress' after filling: {missing_division_address}")

# Print formatted table
print(df[['Division', 'DivisionAddress']].head(10).to_string(index=False))


Missing values in 'Division' after filling: 0
Missing values in 'DivisionAddress' after filling: 0
    Division              DivisionAddress         
  South Division     456 South Ave Austin TX 73301
  North Division 123 North St Springfield IL 62701
  North Division 123 North St Springfield IL 62701
  South Division     456 South Ave Austin TX 73301
   West Division   101 West Dr Sacramento CA 94203
   West Division   101 West Dr Sacramento CA 94203
   West Division   101 West Dr Sacramento CA 94203
   East Division     789 East Blvd Albany NY 12201
   West Division   101 West Dr Sacramento CA 94203
Central Division   202 Main St, Columbus, OH 43085


## **🧱 For the Student to Complete**
Break the Division address up into 4 parts.
1. `DivStreetAddress` "456 South Ave"
2. `DivCity` "Austin"
3. `DivState` "TX"
4. `DivZip` "73301"

Check your work carefully. Add these columns to the dataframe.

In [524]:
# Define a function to clean and process division addresses
def process_division_address(full_address):

    # Remove commas and normalize spaces
    address = re.sub(r'[,\s]+', ' ', full_address.strip())

    # Split division address into parts
    address_parts = address.split()

    # Assign parts based on known length of each element
    zip_code = address_parts[-1] if len(address_parts) > 0 else pd.NA
    state = address_parts[-2] if len(address_parts) > 1 else pd.NA
    city = address_parts[-3] if len(address_parts) > 2 else pd.NA
    street_address = ' '.join(p for p in address_parts[:-3]) if len(address_parts) > 3 else pd.NA

    return pd.Series([street_address, city, state, zip_code])


# Step 2: Apply the function to the 'DivisionAddress' column in the DataFrame
df[['DivStreetAddress', 'DivCity', 'DivState', 'DivZip']] = df['DivisionAddress'].apply(process_division_address)

In [525]:
# check for missing values in newly separated division address columns
missing_street_address = df["DivStreetAddress"].isna().sum()
missing_city = df["DivCity"].isna().sum()
missing_state = df["DivState"].isna().sum()
missing_zip = df["DivZip"].isna().sum()
print(f"Missing values in 'DivStreetAddress' after filling: {missing_street_address}")
print(f"Missing values in 'DivCity' after filling: {missing_city}")
print(f"Missing values in 'DivState' after filling: {missing_state}")
print(f"Missing values in 'DivZip' after filling: {missing_zip}")

print(df[['DivisionAddress', 'DivStreetAddress', 'DivCity', 'DivState', 'DivZip']].head(10).to_string(index=False))

Missing values in 'DivStreetAddress' after filling: 0
Missing values in 'DivCity' after filling: 0
Missing values in 'DivState' after filling: 0
Missing values in 'DivZip' after filling: 0
         DivisionAddress          DivStreetAddress   DivCity   DivState DivZip
    456 South Ave Austin TX 73301  456 South Ave        Austin    TX    73301 
123 North St Springfield IL 62701   123 North St   Springfield    IL    62701 
123 North St Springfield IL 62701   123 North St   Springfield    IL    62701 
    456 South Ave Austin TX 73301  456 South Ave        Austin    TX    73301 
  101 West Dr Sacramento CA 94203    101 West Dr    Sacramento    CA    94203 
  101 West Dr Sacramento CA 94203    101 West Dr    Sacramento    CA    94203 
  101 West Dr Sacramento CA 94203    101 West Dr    Sacramento    CA    94203 
    789 East Blvd Albany NY 12201  789 East Blvd        Albany    NY    12201 
  101 West Dr Sacramento CA 94203    101 West Dr    Sacramento    CA    94203 
  202 Main St, Columb

### **Explanation**

These columns were imputed and standardized from Division to get DivisionAddress and then broken into parts (DivStreetAddress, DivCity, DivState, DivZip). Since a complete and properly formatted address was guaranteed to exist after cleaning/validation steps, it was known that the DivisionAddress would contain all of the necessary elements to fill the other columns. Knowing the formatting, it was known that the Division Zipcode would be the final 'word' in the split address parts. Similarly, the Division State and City were known to be the 2nd and 3rd to last words in the split address, respectively. Lastly, all words from the beginning up until the 3rd to last word (exclusive) had to belong to the street address since all division cities were one word.

### **🏡 Personal Address**

The `PersonalAddress` column contains the addresses of individual employees. These addresses may:
1. Contain unnecessary special characters.
2. Have inconsistent capitalization.
3. Include formatting inconsistencies.

To clean the `PersonalAddress` column, we will:
1. Retain missing values as they are.
2. Remove special characters (except spaces).
3. Normalize capitalization to title case.

#### **🧽 Get Rid of Special Characters and Normalize Capitalization**

We will use the `clean_address` function to process the `PersonalAddress` column, ensuring all addresses are formatted consistently.

In [526]:
df["PersonalAddress"] = df["PersonalAddress"].apply(clean_address)

# Check for missing values in cleaned PersonalAddress
missing_personal_address = df["PersonalAddress"].isna().sum()
print(f"Missing values in 'PersonalAddress': {missing_division_address}")

print(df['PersonalAddress'].head(10).to_string(index=False))

Missing values in 'PersonalAddress': 0
                 929 Beck Course Jamesview TX 86189
     3392 Jared Stravenue North Carmenside IL 81132
             332 Victoria Parkway Sethberg IL 44937
             36066 Munoz Road Campbellland TX 17454
             29967 Patricia Wall Jamesstad CA 70726
       174 Lucas Groves Apt 124 Lesliefurt CA 24625
          213 Brittney Corners North Chris CA 47691
30621 Stanley Springs Apt 328 East Deborah NY 10696
         65024 Michael Dale East Loriville CA 61034
           1658 Andrew Springs South Carly OH 80087


#### **🧪 Validate That Personal Address Is in the Same State as the Division the Person Works For**

In this step, we will:
1. Compare the `PersonalAddress` column with the `DivisionAddress` column to ensure the personal address is in the same state as the division address.
2. Validate each row by:
   - Extracting the state abbreviation from both addresses.
   - Comparing the extracted state abbreviations.
3. Mark each row as:
   - ✅ **Valid**: If the state matches.
   - ❌ **Invalid**: If the state does not match.
   - ⚠️ **Missing**: If one or both addresses are missing.

This ensures consistency between personal and division addresses.

In [527]:
def validate_addresses_match_state(row):
    # check for missing address
    if pd.isna(row["PersonalAddress"]) or pd.isna(row["DivisionAddress"]):
        return "Missing"

    # compare states for division and personal addresses
    personal_state = re.search(STATE_PATTERN, row["PersonalAddress"], flags=re.IGNORECASE)
    division_state = re.search(STATE_PATTERN, row["DivisionAddress"], flags=re.IGNORECASE)

    if personal_state.group(0) == division_state.group(0):
        return "Valid"

    return "Invalid"
    
df["AddressComparison"] = df.apply(validate_addresses_match_state, axis=1)

# count invalid and missing values
invalid_count = (df["AddressComparison"] == "Invalid").sum()
missing_count = (df["AddressComparison"] == "Missing").sum()

print(f"Invalid division and personal addresses: {invalid_count}")
print(f"Missing division or personal addresses: {missing_count}")

Invalid division and personal addresses: 0
Missing division or personal addresses: 72


####  **🔄 If Personal Address Is Missing, Replace with Division Address**

In cases where the `PersonalAddress` is missing, we will:
1. Replace it with the corresponding `DivisionAddress`.
2. Ensure that each row has a valid address wherever possible.

This step ensures that no personal address is left missing if a division address is available.

In [528]:
def fill_missing_personal_address(row):
    # if value exists, return as is
    if not pd.isna(row["PersonalAddress"]):
        return row["PersonalAddress"]

    return row["DivisionAddress"]

df["PersonalAddress"] = df.apply(fill_missing_personal_address, axis=1)

# count missing personal addresses again
missing_personal_count = pd.isna(df["PersonalAddress"]).sum()
print(f"Missing 'PersonalAdddress' after filling: {missing_personal_count}")

print(df['PersonalAddress'].head(10).to_string(index=False))

Missing 'PersonalAdddress' after filling: 0
                 929 Beck Course Jamesview TX 86189
     3392 Jared Stravenue North Carmenside IL 81132
             332 Victoria Parkway Sethberg IL 44937
             36066 Munoz Road Campbellland TX 17454
             29967 Patricia Wall Jamesstad CA 70726
       174 Lucas Groves Apt 124 Lesliefurt CA 24625
          213 Brittney Corners North Chris CA 47691
30621 Stanley Springs Apt 328 East Deborah NY 10696
         65024 Michael Dale East Loriville CA 61034
           1658 Andrew Springs South Carly OH 80087


#### **🧱 Break the `PersonalAddress` into 4 Parts**

1. `PerStreetAddress` "101 West Dr"
2. `PerCity` "Sacramento"
3. `PerState` "CA"
4. `PerZip` "94203"

Check your work carefully. Add these columns to the dataframe.

In [529]:
# create regex pattern for words that are found when city is more than one word (e.g. San Jose, Saint Louis, Fort Worth)
city_first_words = ["New", "North", "South", "East", "West", "Lake", "Saint", "Mount", "Fort", "San"]
COMPOUND_CITY_PATTERN = r'\b(?:' + '|'.join(re.escape(c) for c in city_first_words) + r')\b'

def process_personal_address(personal_address):
    # Remove commas and normalize spaces
    address = re.sub(r'[,\s]+', ' ', personal_address.strip())

    # Split address into individual words
    address_parts = address.split()

    # Assign state and zip code
    zip_code = address_parts[-1] if len(address_parts) > 0 else pd.NA
    state = address_parts[-2] if len(address_parts) > 1 else pd.NA

    # Assign city and street address based on number of words in city
    compound_city = re.search(COMPOUND_CITY_PATTERN, address, flags=re.IGNORECASE)
    if compound_city:
        city = ' '.join(p for p in address_parts[-4:-2]) if len(address_parts) > 3 else pd.NA
        street_address = ' '.join(p for p in address_parts[:-4]) if len(address_parts) > 5 else pd.NA
    else:       
        city = address_parts[-3] if len(address_parts) > 2 else pd.NA
        street_address = ' '.join(p for p in address_parts[:-3]) if len(address_parts) > 4 else pd.NA
    
    return pd.Series([street_address, city, state, zip_code])
    
    
# Step 2: Apply the function to the 'PersonalAddress' column in the DataFrame
df[['PerStreetAddress', 'PerCity', 'PerState', 'PerZip']] = df['PersonalAddress'].apply(process_personal_address)

missing_street_address = df["PerStreetAddress"].isna().sum()
missing_city = df["PerCity"].isna().sum()
missing_state = df["PerState"].isna().sum()
missing_zip = df["PerZip"].isna().sum()

print(f"Missing values in 'PerStreetAddress' after filling: {missing_street_address}")
print(f"Missing values in 'PerCity' after filling: {missing_city}")
print(f"Missing values in 'PerState' after filling: {missing_state}")
print(f"Missing values in 'PerZip' after filling: {missing_zip}")

print(df[['PersonalAddress', 'PerStreetAddress', 'PerCity', 'PerState', 'PerZip']].head(15).to_string(index=False))

Missing values in 'PerStreetAddress' after filling: 0
Missing values in 'PerCity' after filling: 0
Missing values in 'PerState' after filling: 0
Missing values in 'PerZip' after filling: 0
                   PersonalAddress                            PerStreetAddress               PerCity        PerState PerZip
                    929 Beck Course Jamesview TX 86189                929 Beck Course             Jamesview    TX    86189 
        3392 Jared Stravenue North Carmenside IL 81132           3392 Jared Stravenue      North Carmenside    IL    81132 
                332 Victoria Parkway Sethberg IL 44937           332 Victoria Parkway              Sethberg    IL    44937 
                36066 Munoz Road Campbellland TX 17454               36066 Munoz Road          Campbellland    TX    17454 
                29967 Patricia Wall Jamesstad CA 70726            29967 Patricia Wall             Jamesstad    CA    70726 
          174 Lucas Groves Apt 124 Lesliefurt CA 24625       174 Lu

### **Explanation**

After cleaning the personal addresses, formatting was known to be consistent across all addresses. Missing personal addresses were filled with division address as there is no way to impute the exact personal address from the other columns, but an employees' division address would represent a valid address that should be relatively close to where they live. Once all personal addresses were formatted and filled, they could be broken into smaller pieces. Similar to dividing up the division address, the zip code should be the last 'word' and the state should be the second to last word of the full address. The difference in the personal addresses in comparison to division addresses is there is no defined list of valid cities, so this needed to consider the possibility that the city's name could be more than one word (e.g. San Antonio). To identify these scenarios, a short list of words that typically are involved in multi-word city names was defined and a subsequent pattern was constructed. If one of these words was found in the address string, it was presumed that it was part of the city's name, so two words instead of one were extracted to fill PerCity. This also meant that the end of PerStreetAddress was adjusted according to the length of the city's name. Therefore, a mixture of standardizing (for non-empty personal addresses) and imputing was used to fill PersonalAddress, PerStreetAddress, PerCity, PerState, and PerZip

### **📧 Email**

Should be in form, firstname_lastname_number\@KLB.com. If not, please repair.  
#### **🔍 Observations**

1. Capitalization not constant, need to make lower case
2. Sometimes there are honorifics, these should not be there
3. Sometimes the middle name is included, sometimes not.

➡️ Since we know the format required "firstname_lastname_number\@KLB.com", we will simply reconstruct all emails. We need to get the unique number from each. 

Modifications: 
1. From each email, extract the number.
2. Then reconstruct the email as shown above
3. Replace all emails with your newly constructed email.

In [530]:
import random

def reconstruct_emails(row):
    # if missing either first name or last name, return NA
    if pd.isna(row["First Name"]) or pd.isna(row["Last Name"]):
        return pd.NA

    # if email does not currently exist, randomly generate 3 digit number
    if pd.isna(row["Email"]):
        unique_number = str(random.randint(100,999))
    # if email already exists, extract 3 digit ID to use in reformatted email
    else:
        unique_number_match = re.search(r'\d+', row["Email"])
        unique_number = unique_number_match.group(0)

    return f"{row["First Name"].lower()}_{row["Last Name"].lower()}_{unique_number}@KLB.com"

df['Email'] = df.apply(reconstruct_emails, axis=1)
print(df['Email'].head(15).to_string(index=False))

missing_email = df["Email"].isna().sum()
print(f"Missing values in 'Email' after replacing: {missing_email}")

   steven_johnson_145@KLB.com
   robert_schultz_117@KLB.com
       donna_ford_151@KLB.com
 teresa_hernandez_162@KLB.com
christopher_lopez_100@KLB.com
      paul_guerra_188@KLB.com
     martin_brown_143@KLB.com
 catherine_hurley_109@KLB.com
    tony_morrison_111@KLB.com
     michael_cook_148@KLB.com
   kylie_mitchell_177@KLB.com
       martin_cox_155@KLB.com
    kayla_salazar_113@KLB.com
      robert_wong_198@KLB.com
     margaret_lee_191@KLB.com
Missing values in 'Email' after replacing: 0


### **Explanation**

The Email column was imputed from the first name, last name, and email (if not NA) columns to create a standardized format for email addresses. Since the format was known to be firstname_lastname_{3 digit ID}@KLB.com and the name had already been cleaned and broken into parts, this information could easily be found. In the case that email did not have a value already, a random 3 digit number was generated to fill in the ID portion of the email. Otherwise, the number was extracted from the existing email. Imputing the email was easier than trying to standardize because all the parts were already easily accessible (or easily generated) in comparison to adjusting capitalization, removing extra punctuation, etc.

### **☎️ Phone Number**

1. Should be in the form: `(929)614-7016`, where `929` is the area code  
2. If there is an extension, remove it  
3. There is no need for a country code since all employees are in the US  
4. If missing, fill in with the corresponding **division phone number**  
5. When finished, validate that there are **no missing phone numbers**  
6. If the phone is missing, replace it with the division phone number (if available)

In [531]:
division_to_phone = {
    'North Division' : '(217)555-1234',
    'South Division' : '(512)555-6789',
    'East Division' : '(518)555-1234',
    'West Division' : '(916)555-5678',
    'Central Division' : '(916)555-6789'
}

def format_phone_number(row):
    # if phone number is missing, return division phone number
    if pd.isna(row['PhoneNumber']):
        return division_to_phone[row['Division']] or pd.NA

    # remove punctuation
    phone_number = re.sub(r'[().-]', '', row['PhoneNumber'])

    # identify and remove extension (if exists)
    extension_match = re.search('x', phone_number)
    if extension_match:
        phone_number = phone_number[:extension_match.start()]

    # checks for country code (001 or +1)
    if len(phone_number) > 10:
        phone_number = phone_number[-10:]

    # checks for incomplete phone numbers which will be treated same as missing by using division phone number
    if len(phone_number) < 10:
        return division_to_phone[row['Division']] or pd.NA

    return f'({phone_number[:3]}){phone_number[3:6]}-{phone_number[6:]}'

df['PhoneNumber'] = df.apply(format_phone_number, axis=1)
print(df['PhoneNumber'].head(10).to_string(index=False))

missing_phone_number = df["PhoneNumber"].isna().sum()
print(f"Missing values in 'PhoneNumber' after replacing: {missing_phone_number}")


    

(983)309-3096
(696)415-7443
(217)555-1234
(894)497-2204
(859)491-9776
(808)269-2953
(833)539-0926
(814)543-8441
(262)899-8075
(584)843-2102
Missing values in 'PhoneNumber' after replacing: 0


### **Explanation**

In the situation where phone number was filled (with at least 10 digits), only standardization was used. However, in cases where the number was missing or there weren't enough digits to create a valid phone number, it was imputed using the Division column and the phone number known to be associated with that division. For standardization, the first step was removing all punctuation. Then, extensions were identified by checking for the character 'x' and removing everything including and after that character. Next, the last ten digits were retained (or all in the case that it was already the proper length) to get rid of country codes in either the 001 or +1 format. Lastly, the standardized punctuation was added back in to get the format (XXX)XXX-XXXX. The imputed division phone numbers were already in this format, so they did not need to go through the standardization process.

### **🏢 Department**
1. Should be one of the following:
   - `'Human Resources'`, `'Operations'`, `'Engineering'`, `'Administrative'`, `'Executive'`, `'Management'`  
2. If missing, try to infer from the **job title** or **job description**  
   - Use reference file: `job_descriptions.csv`

> **Use `'Human Resources'`, not `'HR'`.** The validator takes the valid departments straight from `job_descriptions.csv`.


In [532]:
# list of keywords that appear in descriptions associated with specific departments
description_to_department = {
    "administrative" : "Administrative",
    "scheduling" : "Administrative",
    "documentation" : "Administrative",
    "communication" : "Administrative",
    "engineering" : "Engineering",
    "technical" : "Engineering",
    "project" : "Engineering",
    "employee" : "Human Resources",
    "recruitment" : "Human Resources",
    "onboarding" : "Human Resources",
    "objectives" : "Management",
    "production" : "Operations",
    "metrics" : "Operations",
    "processes" : "Operations",
    "quality" : "Operations",
    "efficiency" : "Operations",
    "allocation" : "Operations",
    "workflow" : "Operations",
    "challenges" : "Operations",
    "optimization" : "Operations",
    "cost" : "Operations"
}
DESCRIPTION_KEYWORDS = r'\b(' + '|'.join(description_to_department.keys()) + r')\b'

# keywords in title that are associated with specific departments
title_to_department = {
    'chief': 'Executive',
    'division manager': 'Management',
    'human resources': 'Human Resources',
    'hr': 'Human Resources',
    'operations': 'Operations',
    'engineer': 'Engineering',
    'secretary': 'Administrative'
}
TITLE_KEYWORDS = r'\b(' + '|'.join(title_to_department.keys()) + r')\b'

allowed_departments = ["Human Resources", "Operations", "Engineering", "Administrative", "Executive", "Management"]
DEPARTMENTS_PATTERN = r'\b(' + '|'.join(allowed_departments) + r')\b'

def normalize_department(row):
    # if department exists, check if it is in list of valid departments (replacing HR with Human Resources before checking)
    if not pd.isna(row["Department"]):
        normalize_department = re.sub('HR', 'Human Resources', row['Department'])

        valid_department_match = re.search(DEPARTMENTS_PATTERN, normalize_department, flags=re.IGNORECASE)
        if valid_department_match:
            return valid_department_match.group(0).title()

    # uses JobTitle to detect keywords that would match to a specific department
    if not pd.isna(row["JobTitle"]):
        title_match = re.search(TITLE_KEYWORDS, row["JobTitle"], flags=re.IGNORECASE)
        if title_match:
            return title_to_department[title_match.group(0).lower()]

    # matches keywords in job descriptions to a specific department
    if not pd.isna(row["JobDescription"]):
        description_match = re.search(DESCRIPTION_KEYWORDS, row["JobDescription"], flags=re.IGNORECASE)
        if description_match:
            return description_to_department[description_match.group(0).lower()]

    return pd.NA

df['Department'] = df.apply(normalize_department, axis=1)
print(df['Department'].head(15).to_string(index=False))  

missing_department = df["Department"].isna().sum()
print(f"\nMissing values in 'Department' after replacing: {missing_department}")

 Administrative
     Operations
 Administrative
 Administrative
 Administrative
     Operations
     Operations
Human Resources
     Operations
     Operations
     Operations
     Operations
     Operations
     Operations
     Operations

Missing values in 'Department' after replacing: 0


### **Explanation**

A combination of standardization and imputing was applied to clean the Department column. If there is a value in the column that is essentially the same as one of the valid department values (e.g. HR -> Human Resources), then the value will just be standardized with proper capitalization and replacing abbreviations. If the value does not easily map or there is no value, it will be imputed from job title and then job description as a last resort. Certain keywords in the job title were checked for (in an intentional order) to identify the corresponding department for that job title. If that still did not yield a result, keywords from the job descriptions were used in a similar way to try to impute the department. Since the department, job title, and job description are all closely related, this seemed the most reliable way to accurately determine the department if it was missing or invalid.

### **💼 Job Title**

1. Inspect and clean the job title  
2. Make sure the job title matches one of the standard titles  
   - Refer to the `job_descriptions.csv` file  
3. If the title is missing, try to infer it from the **job description**  
4. Ensure the job title is consistent with the assigned **department**

In [533]:
csv_file = "job_descriptions.csv"

try:
    # Attempt to read the CSV file
    jobs = pd.read_csv(csv_file)
    print("✅ Job Descriptions loaded successfully!")

# Handle issues
except FileNotFoundError:
    print(f"❌ Error: The file '{csv_file}' was not found.")

except pd.errors.EmptyDataError:
    print(f"❌ Error: The file '{csv_file}' is empty.")

except pd.errors.ParserError:
    print(f"❌ Error: The file '{csv_file}' contains malformed data.")

except Exception as e:
    print(f"⚠️ An unexpected error occurred: {e}")

✅ Job Descriptions loaded successfully!


In [534]:
job_titles = set(jobs["Job Title"])
JOB_TITLES_PATTERN = r'\b(' + '|'.join(job_titles) + r')\b'

job_descriptions = set(jobs["Job Description"])
JOB_DESCRIPTIONS_PATTERN = r'\b(' + '|'.join(job_descriptions) + r')\b'
JOB_DESCRIPTIONS_PATTERN = re.sub(r"\.", '', JOB_DESCRIPTIONS_PATTERN) # Remove periods as they mess with regex matching

def normalize_job_title(row):
    if not pd.isna(row["JobTitle"]):
        # tries to match job title with titles in job_descriptions.csv
        job_title_match = re.search(JOB_TITLES_PATTERN, row["JobTitle"], flags=re.IGNORECASE)

        # if match is found, capitalize properly and return
        if job_title_match:
            return re.sub("Hr", "HR", job_title_match.group(0).title())

        # since there is mix of HR and Human Resources in titles, sub HR with Human Resources and check title for match again
        job_title_replaced = re.sub("HR", "Human Resources", row["JobTitle"])
        job_title_replaced_match = re.search(JOB_TITLES_PATTERN, job_title_replaced, flags=re.IGNORECASE)

        # if match is found, capitalize properly and return
        if job_title_replaced_match:
            return job_title_replaced_match.group(0).title()

    # if JobTitle is NA or doesn't match job_descriptions.csv titles, search by description
    if not pd.isna(row["JobDescription"]):
        job_description_match = re.search(JOB_DESCRIPTIONS_PATTERN, row["JobDescription"], flags=re.IGNORECASE)

        # if match is found, iterate to find associated title (description capitalization is too inconsistent to search directly)
        # takes first match since some jobs have same description for multiple titles so assume lowest title
        if job_description_match:
            for job_desc in jobs["Job Description"]:
                if job_desc.lower() == job_description_match.group(0).lower() + ".":
                    return jobs[jobs["Job Description"] == job_desc]["Job Title"].iat[0]

    return pd.NA  

df['JobTitle'] = df.apply(normalize_job_title, axis=1)
print(df['JobTitle'].head(15).to_string(index=False))    

missing_job_title = df["JobTitle"].isna().sum()
print(f"Missing values in 'JobTitle' after replacing: {missing_job_title}")

              Secretary Level 2
   Operations Associate Level 4
              Secretary Level 5
              Secretary Level 5
              Secretary Level 6
   Operations Associate Level 5
   Operations Associate Level 4
Human Resources Manager Level 1
   Operations Associate Level 3
        Lead Operations Manager
   Operations Associate Level 4
   Operations Associate Level 2
   Operations Associate Level 3
   Operations Associate Level 3
   Operations Associate Level 9
Missing values in 'JobTitle' after replacing: 1


### **Explanation**

All possible job titles and their corresponding information was loaded from job_descriptions.csv as a reference point for imputing and standardizing job titles. If the JobTitle was found to match a job title listed in the job_descriptions.csv, the capitalization was standardized, and abbreviations (HR vs Human Resources) were made to match whatever the title was listed as in the csv file. If the job title was missing or could not be matched to a job title in the csv file, the job description was attempted to be matched word-for-word with one in the csv file. If a matching job description was found, the associated job title (or lowest level title if more than one matching title existed) was used to fill the JobTitle column. Similar to department, job title and description are closely related so this was the most reliable way to impute the job title when missing or invalid. The lowest level title was chosen as it would be better to underestimate an employee's status rather than overestimate.

### **📝 Job Description**

1. Inspect and clean the job description  
2. Ensure it matches one of the **standard descriptions**  
   - Refer to the `job_descriptions.csv` file for validation

In [535]:
def clean_job_description(row):
    if not pd.isna(row["JobDescription"]):
        # checks job description against job_descriptions.csv
        job_description_match = re.search(JOB_DESCRIPTIONS_PATTERN, row["JobDescription"], flags=re.IGNORECASE)

        # since capitalization in descriptions is inconsistent, iterate to find properly formatted match
        if job_description_match:
            for job_desc in jobs["Job Description"]:
                if job_desc.lower() == job_description_match.group(0).lower() + ".":
                    return job_desc

    # look for match according to job title
    if not pd.isna(row["JobTitle"]):
        job_title_match = re.search(JOB_TITLES_PATTERN, row["JobTitle"])

        # if match, chose description based on division mentioned in description (or first/only option)
        if job_title_match:
            job_descs = jobs[jobs["Job Title"] == row["JobTitle"]]["Job Description"]
            if len(job_descs) == 1 or pd.isna(row['Division']):
                return job_descs.iat[0]
            for desc in job_descs:
                if re.search(row['Division'], desc):
                    return desc

    return pd.NA

df['JobDescription'] = df.apply(clean_job_description, axis=1)
print(df['JobDescription'].head(15).to_string(index=False))    

missing_job_description = df["JobDescription"].isna().sum()
print(f"Missing values in 'JobDescription' after replacing: {missing_job_description}")

Provides administrative support within the South Division, assisting with scheduling, documentation, and communication tasks.
           Performs production and operational tasks within the North Division, ensuring quality and efficiency in processes.
Provides administrative support within the North Division, assisting with scheduling, documentation, and communication tasks.
Provides administrative support within the South Division, assisting with scheduling, documentation, and communication tasks.
 Provides administrative support within the West Division, assisting with scheduling, documentation, and communication tasks.
            Performs production and operational tasks within the West Division, ensuring quality and efficiency in processes.
            Performs production and operational tasks within the West Division, ensuring quality and efficiency in processes.
         Provides strategic workforce planning, manages employee performance, and oversees HR policies as HR Manager L

### **Explanation**

Similar to JobTitle, JobDescription was cleaned using a mixture of standardization and imputing. In the case that the job description could be matched to one in job_descriptions.csv, the exact version found in the csv was used to replace whatever version existed in JobDescription to ensure consistent capitalization and punctuation. If a JobDescription did not yield a match or was missing altogether, JobTitle was used to find the corresponding job description(s). In many cases, there was more than one description for a given job title, each referencing a different division. In this scenario, the Division column was also referenced to select the description with the correct division to ensure consistency. The choice to impute and the columns referenced was once again due to an effort to maximize accuracy and consistency across closely related columns.

### **✅ Perform Validation Steps**

1. Check if the employee’s `JobDescription` exists in the CSV  
2. Check if the employee’s `JobTitle` exists in the CSV  
3. Ensure `JobTitle` and `JobDescription` are consistent with each other  
4. Parse `JobDescription` for division references and validate against the employee’s `Division`  
5. Report all findings

In [536]:
departments = jobs['Department']
DEPARTMENTS_PATTERN = r'\b(' + '|'.join(departments) + r')\b'

divisions = ['North Division', 'East Division', 'South Division', 'West Division', 'Central Division']
DIVISIONS_PATTERN = r'\b(' + '|'.join(divisions) + r')\b'

def validate_job(row):
    # checks for missing values
    if pd.isna(row['Department']) or pd.isna(row['JobTitle']) or pd.isna(row['JobDescription']):
        return 'Missing'

    # checks each column for no valid match in job_descriptions.csv
    departments_match = re.search(DEPARTMENTS_PATTERN, row['Department'])
    if not departments_match:
        return 'Not Found'

    job_title_match = re.search(JOB_TITLES_PATTERN, row['JobTitle'])
    if not job_title_match:
        return 'Not Found'

    job_description_match = re.search(JOB_DESCRIPTIONS_PATTERN, row['JobDescription'])
    if not job_description_match:
        return 'Not Found'

    # checks job description matches job title
    if row['JobDescription'] not in jobs[jobs["Job Title"] == row["JobTitle"]]["Job Description"].values:
        return 'Invalid'

    # checks division in description matches division column
    job_division_match = re.search(DIVISIONS_PATTERN, row['JobDescription'])
    if job_division_match and job_division_match.group(0) != row['Division']:
        return 'Invalid'

    return 'Valid'

df['JobValidation'] = df.apply(validate_job, axis=1)

# count invalid, missing, and non-csv values
invalid_count = (df["JobValidation"] == "Invalid").sum()
missing_count = (df["JobValidation"] == "Missing").sum()
not_found_count = (df["JobValidation"] == "Not Found").sum()

print(f"Invalid job details across categories: {invalid_count}")
print(f"Missing job department, title, or description: {missing_count}")
print(f"Job department, title, or description not found in csv: {not_found_count}")

Invalid job details across categories: 7
Missing job department, title, or description: 1
Job department, title, or description not found in csv: 0


#### **🧼 Please Finish the Following Steps**

1. Inspect the data and make **observations** (in a markdown cell) about what needs to be done  
2. Remove any **extraneous characters**  
3. Normalize **capitalization** across all fields  
4. Perform any other required **cleaning tasks**  
5. **Save** the cleaned data  
6. **Validate** the cleaned data to ensure integrity

The specific issues identified above will need to be investigate more thoroughly to determine the exact source of the issue and create a plan of action accordingly.

### **🔎 Job Title / Description Validity Check**

1. For `MISSING_JOB_DESCRIPTIONS`, use the CSV file to fill in  
2. For `MISSING_JOB_TITLES`, use the CSV file to fill in  
3. For `INCONSISTENT_JOB_TITLE_DESCRIPTION`, assume the **job title** is correct and fix the **job description**  
4. For `INCONSISTENT_DIVISIONS`, assume the **Division** entry is correct and fix the **job description**  
5. Then, rerun the validation step and regenerate the **DETAILED VALIDATION REPORT**

In [537]:
# check problematic rows for specific issue
print(df[df["JobValidation"] == "Missing"]) 
# Since both job title and description are missing, can't reasonably infer a title/description

     EmployeeID           Name           HighestDegree      Division            DivisionAddress         DivisionPhone                  PersonalAddress                              Email              PhoneNumber    Department                          DepartmentDescription                           JobTitle JobDescription   Salary  Birthdate SocialSecurity Honorific First Name Middle Name Last Name Degree DivStreetAddress  DivCity  DivState DivZip AddressComparison       PerStreetAddress        PerCity    PerState PerZip JobValidation
9043  221765.0   BRITTANY MONICA SANDERS       HS       Central Division  202 Main St Columbus OH 43085  (916)555-6789  003 William Unions Apt 341 Parkchester OH 84915  brittany_sanders_177@KLB.com  (439)586-6814  Operations  Manage all purchasing, manufacturing, and logistics of the organization    NaN         NaN       29304.32  9/4/1973   673-43-5379             Brittany     Monica     Sanders           202 Main St    Columbus    OH     43085        Vali

In [538]:
# check problematic rows for specific issue
print(df[df["JobValidation"] == "Invalid"]) # All involve mismatch in division and description

      EmployeeID              Name             HighestDegree      Division              DivisionAddress           DivisionPhone                    PersonalAddress                               Email               PhoneNumber    Department                          DepartmentDescription                                      JobTitle                                                               JobDescription                                                     Salary   Birthdate  SocialSecurity Honorific First Name Middle Name  Last Name Degree DivStreetAddress   DivCity    DivState DivZip AddressComparison        PerStreetAddress            PerCity      PerState PerZip JobValidation
678    283914.0          Michael Jeffrey Braun        HS        South Division      456 South Ave Austin TX 73301  (512)555-6789   516 William Mission Suite 732 Aliciastad TX 76012      michael_braun_186@KLB.com  (512)555-6789  Operations  Manage all purchasing, manufacturing, and logistics of the organization

In [539]:
def align_division_and_description(row):
    if row["JobValidation"] != "Invalid":
        return row["JobDescription"]

    # replace division in description with division column value
    return re.sub(DIVISIONS_PATTERN, row["Division"], row["JobDescription"])

# run alignment and rerun validation
df["JobDescription"] = df.apply(align_division_and_description, axis=1)
df["JobValidation"] = df.apply(validate_job, axis=1)

# count invalid, missing, and non-csv values
invalid_count = (df["JobValidation"] == "Invalid").sum()
missing_count = (df["JobValidation"] == "Missing").sum()
not_found_count = (df["JobValidation"] == "Not Found").sum()

print(f"Invalid job details across categories: {invalid_count}")
print(f"Missing job department, title, or description: {missing_count}")
print(f"Job department, title, or description not found in csv: {not_found_count}")

Invalid job details across categories: 0
Missing job department, title, or description: 1
Job department, title, or description not found in csv: 0


### **Explanation**

The cleaning and validation steps that ultimately revised JobDescription to reference the same division listed in the Division column was to ensure consistency. The choice to treat Division as the source of truth was because it takes a lot more intention to specify division and the corresponding information (e.g. address) than to potentially choose a description that references the wrong (or maybe even an outdated) division. There was one row that was left alone with missing values because it was missing both job title and description. The department was Operations which has numerous different titles and levels, and it would be impossible to accurately deduce the correct title and description without more information.

### **🆔 EmployeeID**

1. Make sure each ID is **cleaned**  
2. Add a column to the dataframe called `'InvalidID'`  
3. Check for **duplicates**  
   - If found, replace all with a new unique ID **within that employee's department range** (see `department_ID_ranges` below)  
   - Add `"duplicate"` to the `'InvalidID'` column  
4. Check for appropriate **ID range**  
   - If an ID is out of range, add `"range"` to the `'InvalidID'` column  
5. Check for **missing values**  
   - If missing, generate a new unique ID **within that employee's department range** (see `department_ID_ranges` below)  
   - Add `"missing"` to the `'InvalidID'` column  
6. If the ID is valid, add `"valid"` to the `'InvalidID'` column

You can use the two dictionaries below if needed:

**Define department ID ranges** <br>
department_ID_ranges = {<br>
    &nbsp;&nbsp;&nbsp;&nbsp;'Human Resources': (100000, 199999),<br>
    &nbsp;&nbsp;&nbsp;&nbsp;'Operations': (200000, 299999),<br>
    &nbsp;&nbsp;&nbsp;&nbsp;'Engineering': (300000, 399999),<br>
    &nbsp;&nbsp;&nbsp;&nbsp;'Administrative': (400000, 499999),<br>
    &nbsp;&nbsp;&nbsp;&nbsp;'Executive': (500000, 599999),<br>
    &nbsp;&nbsp;&nbsp;&nbsp;'Management': (500000, 599999)<br>
}<br>

**Define JobTitle keyword to department mapping** (check the keywords **in this order and stop at the first match** — e.g. *Chief Operations Officer* is Executive, not Operations)<br>
jobtitle_to_department = {<br>
    &nbsp;&nbsp;&nbsp;&nbsp;'Chief': 'Executive',<br>
    &nbsp;&nbsp;&nbsp;&nbsp;'Division Manager': 'Management',<br>
    &nbsp;&nbsp;&nbsp;&nbsp;'Human Resources': 'Human Resources',<br>
    &nbsp;&nbsp;&nbsp;&nbsp;'HR': 'Human Resources',<br>
    &nbsp;&nbsp;&nbsp;&nbsp;'Operations': 'Operations',<br>
    &nbsp;&nbsp;&nbsp;&nbsp;'Engineer': 'Engineering',<br>
    &nbsp;&nbsp;&nbsp;&nbsp;'Secretary': 'Administrative'<br>
}


In [540]:
# valid department ID ranges
department_ID_ranges = {
    'Human Resources': (100000, 199999),
    'Operations': (200000, 299999),
    'Engineering': (300000, 399999),
    'Administrative': (400000, 499999),
    'Executive': (500000, 599999),
    'Management': (500000, 599999)
}

# IDs in use for generating unique IDs
department_ID_used = {
    'Human Resources': set(df[df['Department'] == 'Human Resources']['EmployeeID']),
    'Operations': set(df[df['Department'] == 'Operations']['EmployeeID']),
    'Engineering': set(df[df['Department'] == 'Engineering']['EmployeeID']),
    'Administrative': set(df[df['Department'] == 'Administrative']['EmployeeID']),
    'Executive': set(df[df['Department'] == 'Executive']['EmployeeID'] + df[df['Department'] == 'Management']['EmployeeID']),
    'Management': set(df[df['Department'] == 'Executive']['EmployeeID'] + df[df['Department'] == 'Management']['EmployeeID'])
}

In [541]:
def validate_employee_id(row):
    # check for missing ID
    if pd.isna(row['EmployeeID']):
        return "missing"

    # check for duplicate ID
    if row['EmployeeID'] in set(df[df['EmployeeID'].duplicated()]['EmployeeID']):
        return "duplicate"

    # check for ID out of valid department range
    if row["EmployeeID"] < department_ID_ranges[row['Department']][0] or row["EmployeeID"] > department_ID_ranges[row['Department']][1]:
        return "range"

    return "valid"

df["InvalidID"] = df.apply(validate_employee_id, axis=1)

# count missing, duplicate, and range
missing_count = (df["InvalidID"] == "missing").sum()
duplicate_count = (df["InvalidID"] == "duplicate").sum()
range_count = (df["InvalidID"] == "range").sum()

print(f"Missing employee ID count: {missing_count}")
print(f"Duplicate employee ID count: {duplicate_count}")
print(f"Employee IDs out of range count: {range_count}")

Missing employee ID count: 80
Duplicate employee ID count: 1230
Employee IDs out of range count: 31


In [542]:
def clean_employee_id(row):
    # if already valid, return as is
    if row['InvalidID'] == "valid":
        return row['EmployeeID']

    # replace duplicate, missing, or out of range IDs with valid ID
    # chooses first available ID in range
    for i in range(department_ID_ranges[row['Department']][0], department_ID_ranges[row['Department']][1] + 1):
        if i not in department_ID_used[row['Department']]:

            # adds ID to used sets to mark as now in use
            department_ID_used[row['Department']].add(i)
            if row['Department'] == 'Executive':
                department_ID_used['Management'].add(i)
            elif row['Department'] == 'Management':
                department_ID_used['Executive'].add(i)

            return i

    return pd.NA

df["EmployeeID"] = df.apply(clean_employee_id, axis=1)

# rerun ID validation
df["InvalidID"] = df.apply(validate_employee_id, axis=1)

# count missing, duplicate, and range
missing_count = (df["InvalidID"] == "missing").sum()
duplicate_count = (df["InvalidID"] == "duplicate").sum()
range_count = (df["InvalidID"] == "range").sum()

print(f"Missing employee ID count: {missing_count}")
print(f"Duplicate employee ID count: {duplicate_count}")
print(f"Employee IDs out of range count: {range_count}")

Missing employee ID count: 0
Duplicate employee ID count: 0
Employee IDs out of range count: 0


### **Explanation**

This was one of the only cases where only imputing was necessary since employee ID doesn't necessarily follow a specific format like SSN or phone numbers, so standardization was not necessary. Imputing was applied to the EmployeeID column by using the Department column to validate that existing IDs fell into the proper range for their department and generate IDs as necessary. All existing IDs were checked to make sure that there were no duplicate employee IDs as IDs inherently are used as unique identifiers. Any duplicate IDs were replaced with new, unique values that were in-range for the department specified in the  Department column. Existing, non-duplicate IDs were also replaced with a unique, in-range value if they were outside the valid range for the department. Lastly, missing values were treated similarly with a new unique value generated within the range for the department.

### **💰 Salary**

1. Check whether the salary is within the specified range for the corresponding **job title**  
   - Refer to the `salary_ranges.csv` file  
2. If the salary is **out of range** or **missing**, set it to the **average salary** for that job title

In [543]:
csv_file = "salary_ranges.csv"

try:
    # Attempt to read the CSV file
    salaries = pd.read_csv(csv_file)
    print("✅ Salary ranges loaded successfully!")

except FileNotFoundError:
    print(f"❌ Error: The file '{csv_file}' was not found.")

except pd.errors.EmptyDataError:
    print(f"❌ Error: The file '{csv_file}' is empty.")

except pd.errors.ParserError:
    print(f"❌ Error: The file '{csv_file}' contains malformed data.")

except Exception as e:
    print(f"⚠️ An unexpected error occurred: {e}")

✅ Salary ranges loaded successfully!


In [544]:
def clean_salary(row):
    # if job title does not exist, return salary as is since can't validate/impute without title
    if pd.isna(row['JobTitle']):
        return row['Salary']

    # generate titles that match title format in salary ranges file (all HR, no Human Resources)
    temp_title = re.sub("Human Resources", "HR", row['JobTitle'])

    # if salary is NA, replace with average for job title
    if pd.isna(row['Salary']):
        return salaries[salaries['Job Title'] == temp_title]['Average'].iat[0]

    # check salary is in range for job title
    minRange = salaries[salaries['Job Title'] == temp_title]['Min of Salary'].iat[0]
    maxRange = salaries[salaries['Job Title'] == temp_title]['Max of Salary'].iat[0]

    # if salary is out of range, replace with average
    if row['Salary'] < minRange or row['Salary'] > maxRange:
        return salaries[salaries['Job Title'] == temp_title]['Average'].iat[0]

    return row['Salary']

df['Salary'] = df.apply(clean_salary, axis=1)
print(df['Salary'].head(15).to_string(index=False))  

missing_salary = df['Salary'].isna().sum()
print(f"Missing 'Salary' after cleaning count: {missing_salary}")

   43996.06
   37820.64
   47606.65
   41088.33
    45006.0
   35745.11
   34753.84
  100125.29
   34080.11
  288227.22
   33171.39
   36298.69
   37036.15
   32242.21
38807.11256
Missing 'Salary' after cleaning count: 0


### **Explanation**

The Salary column was treated similarly to the EmployeeID column as only imputing was necessary. Instead of using the Department column like for EmployeeID, Salary relied on the JobTitle to validate and/or replace values. Each salary was checked for falling within the proper range defined in salary_ranges.csv, and out-of-range or missing salaries were replaced with the average salary for that job title. The choice of average as a replacement value for salary was due to the fact that people are generally more likely to have a value near the middle of the range rather than on one extreme. In the case that JobTitle was missing, salary was left as-is since there was no other accurate way of validating/imputing it.

### **🎂 Birthdate**

Ensure that the birthdate is in the proper format: `MM/DD/YYYY`  
📌 Example: `08/01/1960` (two-digit month and day, four-digit year)

**~51 rows have missing Birthdate. You MUST impute them (document your strategy) — leaving them null will exceed the 50-error budget.** Dates appear in exactly two formats: M/D/YYYY and YYYY.MM.DD.


In [545]:
# specifies estimates for years to reach title for age calculation
title_to_years = {
    "Chief": 20,
    "Division Manager": 20,
    "Lead Engineering Manager": 8,
    "Lead HR Manager": 8,
    "Lead Operations Manager": 18,
    "Engineering Manager": 5,
    "HR Manager": 5,
    "Operations Lead": 10,
    "Operations Manager": 15
}
TITLE_EXPERIENCE_PATTERN = r'\b(' + '|'.join(title_to_years.keys()) + r')\b'

# separates degrees by type to better estimate age based on degree length
bachelors_degrees = ['BS', 'BA', 'BSc', 'BFA', 'LLB', 'AB']
masters_degrees = ['MBA', 'MS', 'MA', 'MSc', 'MEd', 'MFA', 'MLA', 'LLM']
doctorate_degrees = ['PhD', 'MD', 'JD', 'DO', 'DDS', 'DVM', 'EdD', 'LLD', 'ScD', 'ThD']

def clean_birth_date(row):
    if pd.isna(row['Birthdate']):
        # estimates years spent in higher education
        edu_years = 0
        if row['Degree'] in bachelors_degrees:
            edu_years = 4
        elif row['Degree'] in masters_degrees:
            edu_years = 6
        elif row['Degree'] in doctorate_degrees:
            edu_years = 10

        # estimates years spent working based on title
        work_years = 0
        if not pd.isna(row['JobTitle']):
            temp_title = re.sub("Human Resources", "HR", row['JobTitle'])

            top_title_match = re.search(TITLE_EXPERIENCE_PATTERN, temp_title)
            if top_title_match:
                work_years = title_to_years[top_title_match.group(0)]

            title_level = re.search(r'\d+', row["JobTitle"])
            if title_level:
                work_years += int(title_level.group(0))

        # calculates birth year using edu_years, work_years, and being at least 18
        return f"01/01/{2026 - 18 - edu_years - work_years}"

    # checks for and handles YYYY.MM.DD format
    period_format_match = re.search(r"\.", row['Birthdate'])
    if period_format_match:
        return f"{row['Birthdate'][-5:-3]}/{row['Birthdate'][-2:]}/{row['Birthdate'][:4]}"

    # if length 8, must be M/D/YYYY so need to add leading zeroes for month and day
    if len(row['Birthdate']) == 8:
        return f"0{row['Birthdate'][:1]}/0{row['Birthdate'][2:]}"
    # if length 10, must by MM/DD/YYYY so return as is
    elif len(row['Birthdate']) == 10:
        return row['Birthdate']

    # find first / to identify if M/DD/YYYY or MM/D/YYYY and add leading zero accordingly
    first_slash_match = re.search(r"\/", row['Birthdate'])
    if first_slash_match:
        if first_slash_match.start() == 1:
            return f"0{row['Birthdate']}"
        return f"{row['Birthdate'][:2]}/0{row['Birthdate'][-6:]}"

    return pd.NA

df['Birthdate'] = df.apply(clean_birth_date, axis=1)
print(df['Birthdate'].head(15).to_string(index=False))   

missing_birthdate = df['Birthdate'].isna().sum()
print(f"Missing 'Birthdate' after cleaning count: {missing_birthdate}")


12/10/1971
01/18/2005
02/13/1965
05/21/1975
10/01/1973
01/16/1983
09/14/1974
08/01/1989
09/21/1993
05/22/1976
06/12/1973
12/30/1970
08/11/1986
04/19/1981
09/03/1999
Missing 'Birthdate' after cleaning count: 0


### **Explanation**

For the Birthdate column, a mixture of standardization and imputing was used. For rows that already had a Birthdate, there were simply reformatted from YYYY.MM.DD or M/D/YYYY to MM/DD/YYYY. In the situation that there was not a birthdate specified, one was imputed from the JobTitle and HighestDegree columns. First of all, it can be assumed that all employees are at least 18 years old, so if this data was collected in 2026 they must all have born on or before 2008. However, many of them possess some level of higher education or have greater seniority in the company that would take time. All degrees are categorized by type (bachelors, masters, doctorate) and each is associated with an average number of years to complete (4, 6, and 10 respectively). In addition to time spent pursuing higher education, there is an obvious indication of hierarchy in the job titles (Level 1, Level 2, ... Lead, Manager, Chief, etc) that would take time to reach. For each percieved promotion, a year of work experience was added/assumed. For the very high level positions (division manager and chiefs), 20 years of work experience was presumed. The years spent in education and the years spent working were both assumed to be completed after the employee had turned 18, so in total their age is estimated to be 18 + education years + work years. Since there is no way to accurately impute an employee's month or day of birth, all employees missing a birthdate were assigned a birthdate of January 1st in whatever birth year would place them at their predicted age in 2026. The reason that I did not also separately factor in military rank (when applicable) is because military experience often serves as relevant work experience, allowing someone fresh out of the military to get a non-entry level role.

### **🔐 Social Security Number**

Ensure that the Social Security Number is in the proper format: `XXX-XX-XXXX`  
📌 Example: `777-35-0022`

#### **🛠️ Note**
If any SSN is **missing**, generate a **unique ID** in the format `XXX-XX-XXXX`  
➡️ Ensure no duplicates exist in the generated values.

In [546]:
print(f"Starting duplicate count in SSN: {df['SocialSecurity'].duplicated().sum()}")

# formats social security numbers to use - instead of / or spaces (leaves NA as is)
def format_social_security(row):
    if pd.isna(row['SocialSecurity']):
        return pd.NA

    return re.sub(r"[\/\s]+", "-", row['SocialSecurity'])

df['SocialSecurity'] = df.apply(format_social_security, axis=1)

Starting duplicate count in SSN: 47


In [547]:
# identifies duplicate Social Security numbers so they can be replaced
duplicate_ssn = set(df[df['SocialSecurity'].duplicated()]['SocialSecurity'])

# identifies SSNs already in use so can generate unique SSNs
used_ssn = set(df['SocialSecurity'])

def change_social_security(row):
    # if SSN exists and is not a duplicate, return as is
    if not pd.isna(row['SocialSecurity']) and row['SocialSecurity'] not in duplicate_ssn:
        return row['SocialSecurity']

    # generate random SSN and format accordingly
    random_num = str(random.randint(100000000, 999999999))
    unique_ssn = f"{random_num[:3]}-{random_num[3:5]}-{random_num[5:]}"

    # continues generating new SSN until creates unique value
    # compares against used_ssn set
    while unique_ssn in used_ssn:
        random_num = random.randint(100000000, 999999999)
        unique_ssn = f"{random_num[0:3]}-{random_num[3:5]}-{random_num[5:]}"

    # adds generated SSN to set to avoid generating a duplicate later on
    used_ssn.add(unique_ssn)
    return unique_ssn

df['SocialSecurity'] = df.apply(change_social_security, axis=1)
print(df['SocialSecurity'].head(15).to_string(index=False))
print(f"Ending duplicate count in SSN: {df['SocialSecurity'].duplicated().sum()}")    


149-51-0794
097-83-2017
494-41-0582
421-73-2111
343-92-4420
002-93-2875
136-76-2413
896-85-7974
362-66-9854
907-23-5911
191-27-3004
113-70-8907
789-58-9775
044-54-5692
058-53-8003
Ending duplicate count in SSN: 0


### **Explanation**

The SocialSecurity column was entirely cleaned with standardization as either the employee possessed a SSN that simply needed to be reformatted or they did not have one, resulting in one being randomly generated for them. Those with duplicate SSNs also had to have a new, unique SSN generated as SSNs should be unique to an individual. None of the other columns in the dataset would be able to inform a more accurate SSN than just a randomly generated one, making imputing not worthwhile/possible.

In [548]:
# uses isolated name columns and combines them back together into the 'Name' column
def format_name(row):
    # If first, middle, and last name all exist, create name with all 3
    if not pd.isna(row['First Name']) and not pd.isna(row['Middle Name']) and not pd.isna(row['Last Name']):
        return f"{row['First Name']} {row['Middle Name']} {row['Last Name']}"

    # if first and last name exist, create name with both (but no middle)
    if not pd.isna(row['First Name']) and not pd.isna(row['Last Name']):
        return f"{row['First Name']} {row['Last Name']}"

    return pd.NA

df['Name'] = df.apply(format_name, axis=1)

# Replaces missing values for HighestDegree with HS
def default_hs_degree(row):
    if pd.isna(row['HighestDegree']):
        return 'HS'

    return row['HighestDegree']

df['HighestDegree'] = df.apply(default_hs_degree, axis=1)

missing_degree = df['HighestDegree'].isna().sum()
print(f"Missing 'HighestDegree' after filling: {missing_degree}")

Missing 'HighestDegree' after filling: 0


In [550]:
# print df info one more time
df.info()

<class 'pandas.DataFrame'>
Index: 14321 entries, 0 to 15242
Data columns (total 32 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   EmployeeID             14321 non-null  float64
 1   Name                   14321 non-null  str    
 2   HighestDegree          14321 non-null  str    
 3   Division               14321 non-null  str    
 4   DivisionAddress        14321 non-null  str    
 5   DivisionPhone          14257 non-null  str    
 6   PersonalAddress        14321 non-null  str    
 7   Email                  14321 non-null  str    
 8   PhoneNumber            14321 non-null  str    
 9   Department             14321 non-null  str    
 10  DepartmentDescription  14267 non-null  str    
 11  JobTitle               14320 non-null  str    
 12  JobDescription         14320 non-null  str    
 13  Salary                 14321 non-null  object 
 14  Birthdate              14321 non-null  str    
 15  SocialSecurity    

### **Explanation**

Some final standardization needed to take place to pass the validator checks. Both the Name and HighestDegree columns were mostly cleaned/validated for us, but the standardized name components needed to be combined back together into the Name column and some NA values in HighestDegree needed to be filled. I decided to fill in all NA values for HighestDegree with HS as there was no other information that wasn't already taken into consideration that could be used to impute a more accurate value (e.g. Honorific). There are very few people in the US that do not hold a HS/GED degree that work an office job, so the assumption that every employee had at least graduated HS did not seem like a massive stretch.

## **💾 Save DataFrame to CSV**

Save the cleaned DataFrame as **`Cleaned_data.csv`** in this folder, e.g. `df.to_csv("Cleaned_data.csv", index=False)`.

The validator requires these 24 columns, spelled exactly like this (extra working columns are reported but not counted as errors):

`EmployeeID`, `Name`, `HighestDegree`, `Division`, `DivisionAddress`, `DivisionPhone`, `DivStreetAddress`, `DivCity`, `DivState`, `DivZip`, `PersonalAddress`, `PerStreetAddress`, `PerCity`, `PerState`, `PerZip`, `Email`, `PhoneNumber`, `Department`, `DepartmentDescription`, `JobTitle`, `JobDescription`, `Salary`, `Birthdate`, `SocialSecurity`

`Name` must be the cleaned full name (title case, no honorific or degree).

📌 We will use this cleaned dataset in the next assignments.

In [ ]:
# save dataframe to csv with only the required columns
df.to_csv("Cleaned_data.csv", index=False, columns = ['EmployeeID', 'Name', 'HighestDegree', 'Division',
                                                        'DivisionAddress', 'DivisionPhone', 'DivStreetAddress',
                                                        'DivCity', 'DivState', 'DivZip', 'PersonalAddress',
                                                        'PerStreetAddress', 'PerCity', 'PerState', 'PerZip',
                                                        'Email', 'PhoneNumber', 'Department', 'DepartmentDescription',
                                                        'JobTitle', 'JobDescription', 'Salary', 'Birthdate', 'SocialSecurity'])

# **Reflection**

Overall, this assignment demonstrated the value and difficulty of standardizing and imputing data. Some columns were fairly easy to clean because they were mostly just reformatting, and missing values were replaced with either randomly generated or easily imputed values. For example, SocialSecurity could just be replaced with a randomly generated, formatted 9 digit number. EmployeeID, Salary, and Email also were pretty easy to fill because the columns they were imputing from were easily mapped to a value.

Other columns were not as easy to impute or reformat. I struggled with figuring out a reliable strategy for dividing PersonalAddress into PerStreetAddress and PerCity because there was no punctuation to differentiate the end of the street address and start of city name. Instead, I had to determine words that would indicate the start of a city name in order to figure out where to separate the components. Other columns that were more difficult to clean were JobTitle and JobDescription, but it was more from an implementation standpoint than a strategic perspective. JobDescription had a lot of inconsistent capitalization in the properly formatted descriptions provided in job_description.csv that made matching and standardizing more difficult. In the case of filling JobTitle using JobDescription, I could not easily search by a matched job description to find the associated title because the re.search match returns the raw text. This raw text could not be converted to the format found in job_descriptions.csv with a simple title() or capitalize() function since there were division or title names capitalized mid-sentence. This became a problem again when trying to save the properly formatted version of JobDescription to the column.

Despite the struggles, the data looks much cleaner in regards to consistent formatting and fewer missing/invalid values. However, not all of the previously missing values represent values that are significantly more meaningful since they were randomly generated or required major guesswork. For example, the randomly generated SocialSecurity values really don't provide any more value in comparison to a null value (at least in my opinion). However, they do make the data easier to work with, so this is one tradeoff that has to be considered.

Another limitation is the possibility for mistakes to be made in the cleaning of some columns, particularly in the division of PersonalAddress. Although I did my best to identify cases where the city name is more than one word, there could be three word cities or cities whose name is two words but not included one of the words I identified. This could mean that the divided address may still be wrong in some cases, but this would be hard to identify and fix without examining the whole dataset.

Another important tradeoff is involved in the decision to replace all duplicate instances for SSN and employee ID with a new, randomly generated value. While I understand that these values are inherently meant to be unique, there is a risk that by replacing all duplicate values with an artificial value you would be taking a correct ID/SSN and replacing it with something incorrect. The method chosen ultimately comes down to whether it is more important to not have duplicate values for columns that shouldn't contain duplicates or to potentially retain more accurate values.